# Sensitivity analysis of the PINN

This notebook trains the Physics-Informed Neural Network (PINN) on the empirical dataset corresponding to dust flux depositions for the Holocene and Last Glacial Maximum. Different values of the weight parameter for the model loss are taken to study the paramter sensitivity of the PINN.

The preprocessing codes should have been performed before.

The training of the PINN may take several hours, depending on the computer facilities.

## Preliminaries

Import the necessary libraries and specify the data folders.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import deepxde as dde
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from deepxde.backend import tf
import pickle
import shutil
import os

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.compat.v1.set_random_seed(SEED)
print(f"Random seed set to {SEED}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_PATH = "../Data/"
INPUT_MODEL_PATH = DATA_PATH + "processed_data/"
MODEL_SAVE_PATH = DATA_PATH + "trained_models/"
RESULTS_PATH = DATA_PATH + "model_results/"
if not os.path.exists(RESULTS_PATH):
    os.makedirs(RESULTS_PATH)

In [ ]:
from style_jcp import load_world
world = load_world()

Load functions for training the PINN model.

In [ ]:
with open("functions_training_model.py", 'r') as file:
    content = file.read()

# Execute the content of the .py file
exec(content)

In [ ]:
df_empirical_Holocene = pd.read_csv(INPUT_MODEL_PATH + "df_empirical_Holocene.csv")
df_empirical_LGM = pd.read_csv(INPUT_MODEL_PATH + "df_empirical_LGM.csv")

In [ ]:
df_global_grid = pd.read_csv(INPUT_MODEL_PATH + "df_global_grid.csv")

In [ ]:
df_wind = pd.read_csv(INPUT_MODEL_PATH + "df_wind.csv", usecols=['wind', 'latitude'])

In [ ]:
from functions_training_model import wind_tf_interp

latitude_wind, mean_wind = df_wind['latitude'].values/90, df_wind['wind'].values/df_wind['wind'].max()

def wind_latitude(latitude):
    interpolated = wind_tf_interp(latitude, tf.convert_to_tensor(latitude_wind), tf.convert_to_tensor(mean_wind))
    return interpolated

tf_wind_latitude = tf.function(wind_latitude)

In [ ]:
def training_points(df):
    data_observ_points = dde.data.DataSet(
        X_train=df[['lon', 'lat']].values / 90,
        y_train=df['log_dep_norm'].values.reshape(-1, 1),
        X_test=df[['lon', 'lat']].values / 90,
        y_test=df['log_dep_norm'].values.reshape(-1, 1),
        standardize=False)
    observe_u = dde.icbc.PointSetBC(
        data_observ_points.train_x,
        df['log_dep_norm'].values.reshape(-1, 1),
        component=0)

    return data_observ_points, observe_u


In [ ]:
x_min, x_max = -2.0, 2.0
y_min, y_max = -0.89, 0.89

left_corner = np.array([x_min, y_min])
right_corner = np.array([x_max, y_max])
geometry_rectangle = dde.geometry.geometry_2d.Rectangle(left_corner, right_corner)

In [ ]:
def pde(x, y):
    u = y[:, 0:1]
    D_raw = y[:, 1:2]

    D = tf.math.softplus(D_raw)
    scale_1st = 2.0 / np.pi
    scale_2nd = (2.0 / np.pi) ** 2

    du_x_raw = dde.grad.jacobian(u, x, j=0)
    du_y_raw = dde.grad.jacobian(u, x, j=1)
    du_xx_raw = dde.grad.hessian(u, x, i=0, j=0)
    du_yy_raw = dde.grad.hessian(u, x, i=1, j=1)

    du_x = du_x_raw * scale_1st
    du_y = du_y_raw * scale_1st
    du_xx = du_xx_raw * scale_2nd
    du_yy = du_yy_raw * scale_2nd

    dD_x_raw = dde.grad.jacobian(D, x, j=0)
    dD_y_raw = dde.grad.jacobian(D, x, j=1)

    dD_x = dD_x_raw * scale_1st
    dD_y = dD_y_raw * scale_1st

    K = wind_latitude(x[:, 1:2])
    K = tf.cast(K, tf.float32)
    cos_theta = tf.cos(x[:, 1:2]* np.pi /2)
    tan_theta = tf.tan(x[:, 1:2]* np.pi /2)

    advection = -K * du_x * (1/cos_theta)
    diffusion_main = D * ( (1/(cos_theta**2)) * du_xx + du_yy - tan_theta * du_y )
    diffusion_grad = (1/(cos_theta**2)) * dD_x * du_x + dD_y * du_y

    pde_residual = advection + diffusion_main + diffusion_grad

    grad_D_sq = dD_x**2 + dD_y**2

    return [pde_residual, grad_D_sq]

In [ ]:
def space_boundary_north(x, on_boundary):
    return on_boundary and np.isclose(y_max, x[1])

def space_boundary_south(x, on_boundary):
    return on_boundary and np.isclose(y_min, x[1])

def periodic_boundary(x, domain):
    return domain and (np.isclose(x[0], x_min) or np.isclose(x[0], x_max))

In [ ]:
def train_process(data_observ_points, observe_u, bc_1, bc_2, model_name, pde_weight):
    save_dir = os.path.join(MODEL_SAVE_PATH, model_name)
    os.makedirs(save_dir, exist_ok=True)

    data = dde.data.PDE(
        geometry_rectangle,
        pde,
        [observe_u, periodic_condition, periodic_condition_derivative,
         periodic_D, periodic_D_derivative,
         bc_1, bc_2],
        num_domain=2592,
        num_boundary=216,
        anchors=data_observ_points.train_x,
        train_distribution='uniform'
    )

    neurons = 32
    layer = 5
    layer_size = [2] + [neurons] * layer + [2]
    activation = "selu"
    initializer = "Glorot normal"
    net = dde.maps.FNN(layer_size, activation, initializer)
    model = dde.Model(data, net)

    dde.optimizers.set_LBFGS_options(maxcor=50, ftol=1e-20, maxiter=1e5)

    reg_weight = pde_weight * 0.1 if pde_weight > 0 else 0
    loss_weights = [pde_weight, reg_weight, 10, 0.5, 0.5, 1, 1, 1, 1]

    model.compile("adam", lr=0.00001, external_trainable_variables=[north_mean, south_mean],
                  loss_weights=loss_weights)

    checkpointer = dde.callbacks.ModelCheckpoint(f"{save_dir}/{model_name}.ckpt", verbose=0, period=10000)
    variable = dde.callbacks.VariableValue([north_mean, south_mean], period=10000, filename=f"{save_dir}/variables.dat")

    losshistory, train_state = model.train(epochs=150000, callbacks=[variable, checkpointer])

    if os.path.exists("loss.dat"): shutil.move("loss.dat", f"{save_dir}/loss.dat")
    if os.path.exists("train.dat"): os.remove("train.dat")
    if os.path.exists("test.dat"): os.remove("test.dat")

    return model, variable.get_value(), train_state.best_step

In [ ]:
weights = [0, 0.1, 10, 1000]

with open(RESULTS_PATH + "weights_sensitivity.csv", 'w') as f:
    pd.DataFrame(data=weights, columns=['weights']).to_csv(f, index=False)

In [ ]:
def denormalize_predictions(normalized_values, lat_values, norm_params):

    global_mean = norm_params['global_mean']
    global_std = norm_params['global_std']
    denorm = normalized_values * global_std + global_mean

    if 'bias_corrections' in norm_params and norm_params['bias_corrections']:
        lat_bands = norm_params['lat_bands']
        lat_band_indices = pd.cut(lat_values, bins=lat_bands, include_lowest=True)

        for band_str, correction_info in norm_params['bias_corrections'].items():
            mask = (lat_band_indices.astype(str) == band_str)
            if mask.any():
                denorm[mask] += correction_info['correction'] * global_std

    return denorm


def calculate_save_df(model, df_to_predict, norm_params, path, filename):

    X_input = df_to_predict[['lon', 'lat']].values / 90
    preds = model.predict(X_input)

    if preds.shape[1] == 2:
        U_pred_norm = preds[:, 0]
        D_raw = preds[:, 1]
    else:
        U_pred_norm = preds[:, 0]
        D_raw = np.zeros_like(U_pred_norm) - 10

    D_pred = np.logaddexp(0, D_raw)

    U_pred_denorm = denormalize_predictions(
        U_pred_norm,
        df_to_predict['lat'].values,
        norm_params
    )

    df_to_predict['PINN_log_dep'] = U_pred_denorm
    df_to_predict['PINN_Diffusivity'] = D_pred

    with open(path + filename, 'w') as f:
        df_to_predict.to_csv(f, index=False)
    
    print(f"✓ Saved: {filename}")
    print(f"  Deposition Range: [{U_pred_denorm.min():.3f}, {U_pred_denorm.max():.3f}]")
    print(f"  Diffusivity Range: [{D_pred.min():.4f}, {D_pred.max():.4f}]")

In [ ]:
with open(INPUT_MODEL_PATH + "norm_params_empirical_Holocene.pkl", 'rb') as f:
    norm_params_Holocene = pickle.load(f)

models_list = []
list_name = []

for idx, pde_weight in enumerate(weights):
    tf.compat.v1.reset_default_graph()

    north_mean = dde.Variable(-1.0)
    south_mean = dde.Variable(-2.0)

    bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
    bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)

    periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=0)
    periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=0)

    periodic_D = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=1)
    periodic_D_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=1)

    data_observ_points_Holocene, observe_u_Holocene = training_points(df_empirical_Holocene)

    name = 'model_empirical_Holocene_' + str(idx)
    model_empirical_Holocene, params, best_step = train_process(data_observ_points_Holocene, observe_u_Holocene, bc_1, bc_2, name, pde_weight)

    models_list.append(model_empirical_Holocene)
    list_name.append(name)

In [ ]:
for idx, model in enumerate(models_list):
    calculate_save_df(
        model,
        df_global_grid.copy(),
        norm_params_Holocene,
        RESULTS_PATH,
        "df_pinn_empirical_Holocene_sensitivity_"+str(idx)+".csv"
    )

In [ ]:
with open(INPUT_MODEL_PATH + "norm_params_empirical_LGM.pkl", 'rb') as f:
    norm_params_LGM = pickle.load(f)

models_list = []
list_name = []

for idx, pde_weight in enumerate(weights):
    tf.compat.v1.reset_default_graph()

    north_mean = dde.Variable(-1.0)
    south_mean = dde.Variable(-2.0)

    bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
    bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)

    periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=0)
    periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=0)

    periodic_D = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=1)
    periodic_D_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=1)

    data_observ_points_LGM, observe_u_LGM = training_points(df_empirical_LGM)

    name = 'model_empirical_LGM_' + str(idx)
    model_empirical_LGM, params, best_step = train_process(data_observ_points_LGM, observe_u_LGM, bc_1, bc_2, name, pde_weight)

    models_list.append(model_empirical_LGM)
    list_name.append(name)

In [ ]:
for idx, model in enumerate(models_list):
    calculate_save_df(
        model,
        df_global_grid.copy(),
        norm_params_LGM,
        RESULTS_PATH,
        "df_pinn_empirical_LGM_sensitivity_"+str(idx)+".csv"
    )